In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

import lightgbm as lgb

train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv')
smaple = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv')

In [2]:
print(smaple)

            id  Churn
0       594194      0
1       594195      0
2       594196      0
3       594197      0
4       594198      0
...        ...    ...
254650  848844      0
254651  848845      0
254652  848846      0
254653  848847      0
254654  848848      0

[254655 rows x 2 columns]


In [3]:
target = 'Churn'
id_col = 'id'
X = train.drop(columns=[target])
y = train[target]
X_test = test.copy()

In [4]:
cat_cols = X.select_dtypes(include=['object']).columns
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

y = le.fit_transform(y.astype(str))

train_data = lgb.Dataset(X,label=y)

In [5]:
params = {
    'objective':'binary',
    'metric':'auc',
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "seed": 42,
    "verbose": -1
}

model = lgb.train(params,train_data,num_boost_round = 1000)

In [6]:
test_preds = model.predict(X_test)
d = (test_preds > 0.5).astype(int)
submission =pd.DataFrame({
    id_col: test[id_col],
    target: d
})
print(submission)

            id  Churn
0       594194      0
1       594195      0
2       594196      0
3       594197      0
4       594198      0
...        ...    ...
254650  848844      0
254651  848845      1
254652  848846      0
254653  848847      0
254654  848848      0

[254655 rows x 2 columns]


In [7]:
submission.to_csv("submission.csv", index=False)
print("Submission saved!")

Submission saved!
